# Phase 1: Data Ingestion & Coverage Analysis

Comprehensive exploration of D-PLACE, Seshat, and DRH data for shamanism research.
- Load all three sources
- Analyze coverage by geography, time period, and variables
- Identify gaps for Phase 2 data collection

In [7]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("Imports successful ✓")

Imports successful ✓


## 1. Data Loading

In [8]:
# Add project root to path
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import parsers
from src.ingest.dplace import parse_dplace
from src.ingest.seshat import parse_seshat
from src.ingest.drh import parse_drh

# Load D-PLACE (real data)
print("Loading D-PLACE...")
dplace_df = parse_dplace(
    societies_path=Path("data/raw/dplace/societies.csv"),
    variables_path=Path("data/raw/dplace/variables.csv"),
    data_path=Path("data/raw/dplace/data.csv"),
)
print(f"  ✓ D-PLACE: {len(dplace_df):,} records")

# Load Seshat (mock or real)
print("Loading Seshat...")
seshat_df = parse_seshat()
print(f"  ✓ Seshat: {len(seshat_df):,} records (mock fallback used)")

# Load DRH (real sample)
print("Loading DRH...")
drh_df = parse_drh(source_path=Path("data/raw/drh/drh_sample.csv"))
print(f"  ✓ DRH: {len(drh_df):,} records")

# Combine
combined_df = pd.concat([dplace_df, seshat_df, drh_df], ignore_index=True)
print(f"\n✓ COMBINED: {len(combined_df):,} records from 3 sources")

Loading D-PLACE...


FileNotFoundError: Could not load D-PLACE CSV files: [Errno 2] No such file or directory: 'data/raw/dplace/societies.csv'

## 2. Overall Statistics

In [ ]:
print("\n" + "="*70)
print("OVERALL STATISTICS")
print("="*70)

print(f"\nTotal Records: {len(combined_df):,}")
print(f"Unique Cultures: {combined_df['culture_id'].nunique():,}")
print(f"Unique Variables: {combined_df['variable_name'].nunique()}")
print(f"\nSources:")
for source in combined_df['source'].unique():
    count = len(combined_df[combined_df['source'] == source])
    cultures = combined_df[combined_df['source'] == source]['culture_id'].nunique()
    print(f"  {source.upper():10} {count:6,} records  {cultures:4,} cultures")

print(f"\nTime Period Coverage:")
print(f"  Min: {combined_df['time_start'].min():.0f}")
print(f"  Max: {combined_df['time_end'].max():.0f}")
print(f"  Range: {combined_df['time_end'].max() - combined_df['time_start'].min():.0f} years")

## 3. Geographic Coverage

In [ ]:
# Geographic bounds
print("\n" + "="*70)
print("GEOGRAPHIC COVERAGE")
print("="*70)

gdf = combined_df[combined_df['lat'].notna() & combined_df['lon'].notna()]

print(f"\nRecords with coordinates: {len(gdf):,} / {len(combined_df):,} ({100*len(gdf)/len(combined_df):.1f}%)")
print(f"\nLatitude range: {gdf['lat'].min():.1f}° to {gdf['lat'].max():.1f}°")
print(f"Longitude range: {gdf['lon'].min():.1f}° to {gdf['lon'].max():.1f}°")

# By source
print(f"\nGeographic coverage by source:")
for source in combined_df['source'].unique():
    source_gdf = gdf[gdf['source'] == source]
    pct = 100 * len(source_gdf) / len(combined_df[combined_df['source'] == source])
    print(f"  {source.upper():10} {len(source_gdf):6,} / {len(combined_df[combined_df['source'] == source]):6,} ({pct:5.1f}%)")

## 4. Geographic Visualization

In [ ]:
# Scatter plot of all cultures with coordinates
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, source in enumerate(['dplace', 'seshat', 'drh']):
    source_gdf = gdf[gdf['source'] == source]
    if len(source_gdf) > 0:
        ax = axes[idx]
        scatter = ax.scatter(source_gdf['lon'], source_gdf['lat'], 
                            alpha=0.5, s=50, c=source_gdf['lat'], 
                            cmap='viridis')
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_title(f'{source.upper()}\n{len(source_gdf):,} records')
        ax.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=ax, label='Latitude')

plt.tight_layout()
plt.savefig('data/visualizations/01_geographic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Geographic distribution saved")

## 5. Temporal Coverage

In [ ]:
print("\n" + "="*70)
print("TEMPORAL COVERAGE")
print("="*70)

tdf = combined_df[combined_df['time_start'].notna()]
print(f"\nRecords with time information: {len(tdf):,} / {len(combined_df):,} ({100*len(tdf)/len(combined_df):.1f}%)")

# Temporal distribution by source
print(f"\nTemporal range by source:")
for source in combined_df['source'].unique():
    source_df = combined_df[combined_df['source'] == source]
    source_tdf = source_df[source_df['time_start'].notna()]
    if len(source_tdf) > 0:
        print(f"\n  {source.upper()}:")
        print(f"    Min year: {source_tdf['time_start'].min():.0f}")
        print(f"    Max year: {source_tdf['time_start'].max():.0f}")
        print(f"    Range: {source_tdf['time_start'].max() - source_tdf['time_start'].min():.0f} years")

## 6. Temporal Distribution Visualization

In [ ]:
# Timeline visualization
fig, ax = plt.subplots(figsize=(14, 6))

colors = {'dplace': '#1f77b4', 'seshat': '#ff7f0e', 'drh': '#2ca02c'}

for source in combined_df['source'].unique():
    source_tdf = tdf[tdf['source'] == source]
    if len(source_tdf) > 0:
        ax.scatter(source_tdf['time_start'], [source]*len(source_tdf), 
                  alpha=0.6, s=100, label=source.upper(), color=colors.get(source, '#d62728'))

ax.set_xlabel('Year (BCE negative, CE positive)')
ax.set_ylabel('Data Source')
ax.set_title('Temporal Distribution of Shamanism Data')
ax.legend(loc='best')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('data/visualizations/02_temporal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Temporal distribution saved")

## 7. Variable Coverage Analysis

In [ ]:
print("\n" + "="*70)
print("VARIABLE COVERAGE")
print("="*70)

print(f"\nTotal unique variables: {combined_df['variable_name'].nunique()}")
print(f"\nTop 20 most frequent variables:")
var_counts = combined_df['variable_name'].value_counts().head(20)
for var, count in var_counts.items():
    print(f"  {var:30} {count:6,} records")

print(f"\nVariables by source:")
for source in combined_df['source'].unique():
    source_df = combined_df[combined_df['source'] == source]
    nvars = source_df['variable_name'].nunique()
    print(f"  {source.upper():10} {nvars:3} variables")

## 8. Variable Type Distribution

In [ ]:
# Variable types
print(f"\nVariable types in dataset:")
var_types = combined_df['variable_type'].value_counts()
for vtype, count in var_types.items():
    pct = 100 * count / len(combined_df)
    print(f"  {vtype:15} {count:6,} ({pct:5.1f}%)")

# By source
print(f"\nVariable types by source:")
var_by_source = pd.crosstab(combined_df['source'], combined_df['variable_type'])
print(var_by_source)

## 9. Data Completeness & Missing Values

In [ ]:
print("\n" + "="*70)
print("DATA COMPLETENESS")
print("="*70)

print(f"\nMissing values by column:")
missing = combined_df.isnull().sum()
for col in combined_df.columns:
    count = missing[col]
    pct = 100 * count / len(combined_df)
    if count > 0:
        print(f"  {col:20} {count:6,} ({pct:5.1f}%)")

print(f"\nValue data quality:")
null_vals = combined_df['variable_value'].isnull().sum()
print(f"  Null values: {null_vals:,} ({100*null_vals/len(combined_df):.1f}%)")
print(f"  Valid values: {len(combined_df) - null_vals:,} ({100*(len(combined_df)-null_vals)/len(combined_df):.1f}%)")

## 10. Cross-Source Overlap Analysis

## 10. Cross-Source Overlap Analysis

In [ ]:
print("\n" + "="*70)
print("CROSS-SOURCE OVERLAP")
print("="*70)

# Culture overlap
dplace_cultures = set(dplace_df['culture_id'].unique())
seshat_cultures = set(seshat_df['culture_id'].unique())
drh_cultures = set(drh_df['culture_id'].unique())

print(f"\nCultures by source:")
print(f"  D-PLACE: {len(dplace_cultures):,}")
print(f"  Seshat:  {len(seshat_cultures):,}")
print(f"  DRH:     {len(drh_cultures):,}")
print(f"  Total unique: {len(dplace_cultures | seshat_cultures | drh_cultures):,}")

# Overlaps
overlap_dp_sh = len(dplace_cultures & seshat_cultures)
overlap_dp_dr = len(dplace_cultures & drh_cultures)
overlap_sh_dr = len(seshat_cultures & drh_cultures)
overlap_all = len(dplace_cultures & seshat_cultures & drh_cultures)

print(f"\nCulture overlaps:")
print(f"  D-PLACE ∩ Seshat: {overlap_dp_sh}")
print(f"  D-PLACE ∩ DRH:    {overlap_dp_dr}")
print(f"  Seshat ∩ DRH:     {overlap_sh_dr}")
print(f"  All three:        {overlap_all}")

## 11. Coverage Gaps & Phase 2 Priorities

print("\n" + "="*70)
print("PHASE 2 DATA COLLECTION PRIORITIES")
print("="*70)

# Geographic gaps
print("\n1. GEOGRAPHIC GAPS:")
gdf_summary = gdf.groupby('source').agg({
    'lat': ['min', 'max', 'count'],
    'lon': ['min', 'max']
})

print(f"  Current coverage: {len(gdf):,} / {len(combined_df):,} records ({100*len(gdf)/len(combined_df):.1f}%)")
print(f"  Priority: Complete geographic data for {len(combined_df) - len(gdf):,} records")

# Temporal gaps
print(f"\n2. TEMPORAL GAPS:")
tdf_summary = tdf.groupby('source').agg({'time_start': ['min', 'max', 'count']})
missing_time = len(combined_df) - len(tdf)
print(f"  Current coverage: {len(tdf):,} / {len(combined_df):,} records ({100*len(tdf)/len(combined_df):.1f}%)")
print(f"  Priority: Assign temporal data to {missing_time:,} records")

# Variable coverage
print(f"\n3. VARIABLE COVERAGE GAPS:")
print(f"  D-PLACE variables: {dplace_df['variable_name'].nunique()}")
print(f"  Seshat variables: {seshat_df['variable_name'].nunique()}")
print(f"  DRH variables: {drh_df['variable_name'].nunique()}")
print(f"  Priority: Standardize variable definitions across sources")

# Source-specific issues
print(f"\n4. SOURCE-SPECIFIC PRIORITIES:")
missing_values_by_source = combined_df.groupby('source')['variable_value'].apply(lambda x: x.isnull().sum())
print(f"  D-PLACE: {missing_values_by_source.get('dplace', 0):,} missing values (data lookup needed)")
print(f"  Seshat: Real data required (currently using mock fallback)")
print(f"  DRH: {missing_values_by_source.get('drh', 0):,} missing values (expand sample coverage)")

## 12. Recommendations Summary

print("\n" + "="*70)
print("RECOMMENDATIONS FOR PHASE 2")
print("="*70)

recommendations = [
    ("1. SESHAT REAL DATA", "Obtain Seshat credentials or download complete public release"),
    ("2. GEOGRAPHIC COMPLETENESS", f"Add coordinates for {len(combined_df) - len(gdf):,} records"),
    ("3. TEMPORAL EXPANSION", f"Extend time period coverage for {missing_time:,} records"),
    ("4. VARIABLE STANDARDIZATION", "Align 75 unique variables to core shamanism constructs"),
    ("5. CROSS-SOURCE VALIDATION", "Verify overlapping cultures have consistent values"),
    ("6. DATA QUALITY AUDITS", "Review missing value codes"),
]

for title, desc in recommendations:
    print(f"\n{title}")
    print(f"  {desc}")

print("\n" + "="*70)
print("PHASE 1 DATA INGESTION STATUS: COMPLETE ✓")
print("="*70)